# Data
Loads DESI spectra from MultimodalUniverse/desi on HuggingFace.
Run the inspection cell first to confirm field names before full loading.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import sys, os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import config

In [ ]:
def inspect_dataset():
    """Fetch one record and print its keys and field shapes to confirm the schema."""
    raw = load_dataset(
        'MultimodalUniverse/desi',
        split='train[:1]',
        trust_remote_code=True,
    )
    print('Features:', raw.features)
    print('Keys:', list(raw[0].keys()))
    sample = raw[0]
    for k, v in sample.items():
        shape = len(v) if isinstance(v, list) else type(v).__name__
        print(f'  {k}: {shape}')

inspect_dataset()

In [ ]:
FLUX_KEY_CANDIDATES = ['spectrum_flux', 'spectrum', 'flux']
Z_KEY_CANDIDATES    = ['redshift', 'Z', 'z']


def _detect_keys(sample):
    """Detect flux and redshift field names from a dataset sample.

    Args:
        sample (dict): A single record from the HuggingFace dataset.

    Returns:
        tuple: (flux_key (str), z_key (str))

    Raises:
        KeyError: If neither a flux nor a redshift field is found.
    """
    flux_key = next((k for k in FLUX_KEY_CANDIDATES if k in sample), None)
    z_key    = next((k for k in Z_KEY_CANDIDATES    if k in sample), None)
    if flux_key is None:
        raise KeyError(f'No flux field found. Available keys: {list(sample.keys())}')
    if z_key is None:
        raise KeyError(f'No redshift field found. Available keys: {list(sample.keys())}')
    print(f'Detected keys — flux: "{flux_key}"  redshift: "{z_key}"')
    return flux_key, z_key

In [ ]:
class DESISpectraDataset(Dataset):
    """PyTorch Dataset wrapping DESI spectra from the Multimodal Universe HuggingFace dataset.

    Args:
        hf_split: HuggingFace dataset split containing flux and redshift fields.
    """

    def __init__(self, hf_split):
        self.data = hf_split
        self.flux_key, self.z_key = _detect_keys(hf_split[0])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Return a single normalized, padded spectrum and its redshift.

        Args:
            idx (int): Sample index.

        Returns:
            tuple: (flux (torch.Tensor, shape PADDED_LEN), redshift (torch.Tensor, scalar))
        """
        item = self.data[idx]
        flux = np.array(item[self.flux_key], dtype=np.float32)
        redshift = float(item[self.z_key])

        flux = self._normalize(flux)
        flux = self._pad(flux)

        return (
            torch.tensor(flux, dtype=torch.float32),
            torch.tensor(redshift, dtype=torch.float32),
        )

    def _normalize(self, flux):
        """Apply median/MAD normalization to a flux array.

        Args:
            flux (np.ndarray): Raw flux array.

        Returns:
            np.ndarray: Normalized flux array.
        """
        flux = np.nan_to_num(flux, nan=0.0, posinf=0.0, neginf=0.0)
        median = np.median(flux)
        mad = np.median(np.abs(flux - median))
        return (flux - median) / (mad + 1e-8)

    def _pad(self, flux):
        """Zero-pad flux from SPECTRUM_LEN to PADDED_LEN.

        Args:
            flux (np.ndarray): Flux array of length SPECTRUM_LEN.

        Returns:
            np.ndarray: Padded flux array of length PADDED_LEN.
        """
        padded = np.zeros(config.PADDED_LEN, dtype=np.float32)
        padded[:len(flux)] = flux[:config.PADDED_LEN]
        return padded

In [ ]:
def get_dataloaders(batch_size=config.BATCH_SIZE, n_train=8000, n_val=1000):
    """Load the Multimodal Universe DESI dataset and return train/val DataLoaders.

    Args:
        batch_size (int): Batch size for both loaders.
        n_train (int): Number of training samples.
        n_val (int): Number of validation samples.

    Returns:
        tuple: (train_loader, val_loader)
    """
    total = n_train + n_val
    raw = load_dataset(
        'MultimodalUniverse/desi',
        split=f'train[:{total}]',
        trust_remote_code=True,
    )

    train_ds = DESISpectraDataset(raw.select(range(n_train)))
    val_ds   = DESISpectraDataset(raw.select(range(n_train, total)))

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True, drop_last=False,
    )

    print(f'Train: {len(train_ds)} samples  Val: {len(val_ds)} samples')
    return train_loader, val_loader